# Lesson 5: Causal self-attention

Lesson 4 ended with a residual-stream vector at each token position. Attention lets each position gather information from earlier positions. This lesson works through one small head with three positions and two channels, so every matrix stays visible.

For an input matrix \(X\), learned projections create queries, keys, and values:

\[Q=XW_Q,\qquad K=XW_K,\qquad V=XW_V.\]

A query is compared with every key. The scaled scores are masked so a position cannot see the future, softmax turns the allowed scores into weights, and those weights mix the value vectors:

\[Y=\operatorname{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}+M\right)V.\]

The toy uses identity projections to focus on the attention calculation. GPT-2 learns separate Q/K/V projection weights, splits its 768 channels into 12 heads of width 64, then joins the heads and applies an output projection. This lesson stops at one head's context vectors; the C and Java files show the scalar loops, while PyTorch exposes its fused operation.

**Read the companions:** [C attention_lesson.c](../../c/src/attention_lesson.c) and [Java AttentionLesson.java](../../java/src/main/java/learning/karpathy/gpt2/AttentionLesson.java). Run them with -Action lesson -Lesson 5.


In [ ]:
import math
import sys

import torch
import torch.nn.functional as F

torch.set_printoptions(precision=4, sci_mode=False)
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__} (device: cpu for this tiny example)")


## 1. Project the residual stream into Q, K, and V

We reuse the first sequence from Lesson 4, so X has shape (T, C) = (3, 2). Each projection is a matrix multiplication. The identity matrices here leave the values unchanged; in a trained model, the three matrices become different learned transformations.

In GPT-2, one linear layer computes all three projections together and then splits the result. Separate matrices make the roles clearer for this first example.


In [ ]:
X = torch.tensor(
    [[1.1, 0.0], [0.0, 1.2], [1.3, 1.3]],
    dtype=torch.float64,
)
W_Q = torch.eye(2, dtype=torch.float64)
W_K = torch.eye(2, dtype=torch.float64)
W_V = torch.eye(2, dtype=torch.float64)

Q = X @ W_Q
K = X @ W_K
V = X @ W_V

print("X (T, C):\n", X)
print("Q = X W_Q:\n", Q)
print("K = X W_K:\n", K)
print("V = X W_V:\n", V)
assert Q.shape == K.shape == V.shape == (3, 2)


## 2. Score keys, prevent looking ahead, and normalize

The dot product Q @ K.T gives one compatibility score for every query/key pair. Dividing by sqrt(d_k) keeps score magnitudes better behaved as the head width grows. The causal mask replaces future scores with negative infinity; their softmax probability is therefore zero.

Rows are query positions and columns are key positions. The upper triangle is masked: position 0 sees only 0, position 1 sees 0–1, and position 2 sees 0–2.


In [ ]:
head_size = Q.shape[-1]
scores = (Q @ K.T) / math.sqrt(head_size)
future_positions = torch.triu(torch.ones_like(scores, dtype=torch.bool), diagonal=1)
masked_scores = scores.masked_fill(future_positions, -torch.inf)
attention_weights = torch.softmax(masked_scores, dim=-1)

print("Scaled scores Q K^T / sqrt(d_k):\n", scores)
print("After causal mask (-inf means not allowed):\n", masked_scores)
print("Attention weights after row-wise softmax:\n", attention_weights)
assert torch.allclose(attention_weights.sum(dim=-1), torch.ones(3, dtype=torch.float64))
assert torch.count_nonzero(torch.triu(attention_weights, diagonal=1)) == 0


## 3. Mix values and compare with PyTorch's attention operation

Each output is a weighted sum of value vectors. PyTorch's scaled_dot_product_attention performs the same operation and applies the causal mask through is_causal=True. The four-dimensional shape (B, heads, T, head_size) matches the layout used by the reference GPT-2 code; here B=1 and heads=1.

The last check changes the final input dramatically. Position 0's result must stay the same because it cannot attend to position 2. This is the causal property that makes next-token prediction possible.


In [ ]:
context = attention_weights @ V

library_context = F.scaled_dot_product_attention(
    Q[None, None, :, :],
    K[None, None, :, :],
    V[None, None, :, :],
    is_causal=True,
)[0, 0]

torch.testing.assert_close(context, library_context, rtol=1e-10, atol=1e-10)
print("Context vectors = attention_weights @ V:\n", context)
print("PyTorch scaled_dot_product_attention matches the explicit calculation:",
      torch.allclose(context, library_context, rtol=1e-10, atol=1e-10))

changed_X = X.clone()
changed_X[2] = torch.tensor([101.3, -98.7], dtype=torch.float64)
changed_context = F.scaled_dot_product_attention(
    (changed_X @ W_Q)[None, None, :, :],
    (changed_X @ W_K)[None, None, :, :],
    (changed_X @ W_V)[None, None, :, :],
    is_causal=True,
)[0, 0]
torch.testing.assert_close(context[0], changed_context[0], rtol=1e-10, atol=1e-10)
print("Changing the final input leaves position 0's output unchanged: yes")


## Connect this to GPT-2

Karpathy's CausalSelfAttention.forward uses one linear layer to produce Q, K, and V, reshapes the channels into 12 heads of width 64, calls F.scaled_dot_product_attention(..., is_causal=True), joins the head outputs, and applies a final projection. Our manual mask and weighted-sum cells expose what that library call does. The PyTorch implementation may choose an appropriate CPU or GPU kernel for the tensor and device.

This is still only the attention sublayer. A GPT-2 block also normalizes its input, adds the attention result back through a residual connection, then runs a feed-forward MLP and another residual connection; those fit more naturally in the next lesson.

**Try it:** replace one identity matrix with a simple non-identity matrix and observe how the scores or value vectors change. Keep the causal assertions passing.

**References:** [Attention Is All You Need](https://arxiv.org/abs/1706.03762), [PyTorch scaled_dot_product_attention](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html), and the [reference CausalSelfAttention source](../../build-nanogpt/train_gpt2.py#L12).
